# Figure 2-1 collection

This notebook retains only four epithelial UMAP/PAGA outputs and two tumor-only marker dotplots.


In [ ]:
from pathlib import Path
import warnings

import matplotlib as mpl
import matplotlib.pyplot as plt
import scanpy as sc

warnings.filterwarnings("ignore")
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["font.family"] = "DejaVu Sans"

ROOT = Path("/mnt/disk18t/lr_xcy/riku/kirc_205")
ADATA_PATH = ROOT / "leiden_detailed/adata_epi.h5ad"
EPITHELIAL_FIG_DIR = ROOT / "leiden_detailed/figures"
TRAJECTORY_FIG_DIR = ROOT / "leiden_detailed/dpt_root_from_miox/figures"
EPITHELIAL_FIG_DIR.mkdir(parents=True, exist_ok=True)
TRAJECTORY_FIG_DIR.mkdir(parents=True, exist_ok=True)

adata_all = sc.read_h5ad(ADATA_PATH)

# 1. Nine-state epithelial UMAP: umap_subtype_epi.pdf
sc.settings.figdir = EPITHELIAL_FIG_DIR
sc.set_figure_params(figsize=(3, 3), dpi=150)
sc.pl.umap(
    adata_all,
    color="cell_subtype",
    title="Epi",
    save="_subtype_epi.pdf",
    show=False,
)
plt.close("all")

# Restrict trajectory analysis to the six connected epithelial states.
adata_all.obsm["X_umap_backup"] = adata_all.obsm["X_umap"].copy()
excluded_states = ["Epi_CD44_high", "Epi_AQP2", "Epi_CA12"]
adata = adata_all[~adata_all.obs["cell_subtype"].isin(excluded_states)].copy()

# Retained only to select the Epi_MIOX DPT root cell; no stemness figure is drawn.
STEMNESS_GENES = [
    "ACLY", "ADAM17", "AKT1", "ALDH1B1", "ALDH1L1", "BMI1", "CA9", "CD24",
    "CD34", "CD44", "CDKN1A", "CDKN2A", "CTBP2", "CXCR4", "DHX9", "DOT1L",
    "E2F2", "EGF", "EPAS1", "EZH2", "FKBP5", "FZD4", "GJA1", "GLI1", "HIF1A",
    "HOTAIR", "ID4", "IGF1R", "IL22", "ITGA6", "JAK2", "KDR", "KLF4", "KLF5",
    "KLF9", "L1TD1", "LGR5", "LILRB2", "LIN28A", "MCAM", "MICU1", "NANOG",
    "NCAM1", "NES", "NOTCH1", "PHLDA2", "PLAU", "POU5F1", "PROM1", "PSMD10",
    "PTEN", "PTTG1", "RAC1", "RECK", "REG4", "S100A4", "SCGB2A1", "SEMA3F",
    "SIRT2", "SMAD6", "SOX2", "SPDYA", "STAT3", "TERT", "TGFB1", "TGFBR2",
    "TP53", "TP63", "TWIST1", "USP7", "YAP1", "YBX1",
]
raw_genes = set(adata.raw.var_names)
stemness_genes_present = [gene for gene in STEMNESS_GENES if gene in raw_genes]
sc.tl.score_genes(
    adata,
    gene_list=stemness_genes_present,
    score_name="stemness_score",
    use_raw=True,
)

sc.tl.paga(adata, groups="cell_subtype")
sc.tl.umap(adata, init_pos="paga")

miox_mask = adata.obs["cell_subtype"] == "Epi_MIOX"
root_name = adata.obs.loc[miox_mask, "stemness_score"].idxmin()
adata.uns["iroot"] = adata.obs_names.get_loc(root_name)
sc.tl.diffmap(adata, n_comps=15)
sc.tl.dpt(adata, n_dcs=10, n_branchings=0)

sc.settings.figdir = TRAJECTORY_FIG_DIR

# 2. PAGA graph: pagadpt.pdf
sc.pl.paga(
    adata,
    color="cell_subtype",
    node_size_scale=5,
    save="dpt.pdf",
    show=False,
)
plt.close("all")

# 3. PAGA-initialized state/DPT UMAP: umap_epi_paga.pdf
sc.pl.umap(
    adata,
    color=["dpt_pseudotime", "cell_subtype"],
    save="_epi_paga.pdf",
    show=False,
)
plt.close("all")

# 4. Original state/DPT UMAP: X_umap_backupumap_dpt_leiden.pdf
sc.pl.embedding(
    adata,
    basis="X_umap_backup",
    color=["dpt_pseudotime", "cell_subtype"],
    save="umap_dpt_leiden.pdf",
    show=False,
)
plt.close("all")


## Tumor-only epithelial marker landscape and paired Epi_JUN–Epi_VIM contrast

Output: `tumor_epithelial_marker_publication_figure_headers_above_panel_frames_equal_y_title_clearance.pdf`.


In [ ]:
"""Generate the two numerical source tables used by the publication marker figure.

Panel A uses tumor-only log-normalized expression from adata_epi.raw, aggregates
sample-state means/detection fractions, retains groups with at least 10 cells,
and fits expression ~ sample + state. Panel B prepares paired tumor pseudobulks from raw counts, requiring at least
20 cells in both Epi_JUN and Epi_VIM. The paired edgeR model itself is kept in
fig2-2.R.
"""

from pathlib import Path

import anndata as ad
import numpy as np
import pandas as pd
from scipy import sparse

PROJECT_ROOT = Path("/mnt/disk18t/lr_xcy/riku/kirc_205")
PANEL_A_H5AD = PROJECT_ROOT / "leiden_detailed/adata_epi.h5ad"
PANEL_B_H5AD = PROJECT_ROOT / "ccrcc_cellnavi/data/ccrcc_epi_jun_to_epi_vim_raw_counts.h5ad"
MIN_A = 10
MIN_B = 20
A_STATES = ["Epi_MIOX","Epi_GPX3","Epi_ALDOB","Epi_CA9","Epi_JUN","Epi_VIM","Epi_AQP2","Epi_CA12"]
A_PROGRAMS = {
    "Epi_MIOX":["MIOX","PCK1","NAT8","CUBN"],
    "Epi_GPX3":["GPX3","GATM","ASS1","PDZK1IP1"],
    "Epi_ALDOB":["ALDOB","FABP1","APOE","HPD"],
    "Epi_CA9":["CA9","NDUFA4L2","EGLN3","NDRG1"],
    "Epi_JUN":["KLF6","ZBTB20","SOD2","VMP1","FOSB","JUN"],
    "Epi_VIM":["VIM","LGALS1","LGALS3","NUPR1","S100A6","S100A4"],
    "Epi_AQP2":["AQP2","FXYD4","HSD11B2","DEFB1"],
    "Epi_CA12":["CA12","ATP6V0A4","FOXI1","ATP6V1G3"],
}
B_GROUPS = {
    "AP-1 / immediate-early":["EGR1","FOSB","JUN","FOS","ATF3","IER2"],
    "JUN-associated, non-IEG":["KLF6","ZBTB20","SPP1","ANGPTL4","SOD2","VEGFA","VCAM1","CXCL14"],
    "VIM-associated":["VIM","LGALS1","S100A6","NUPR1","LGALS3","S100A4"],
}


def aggregate_sample_state(x, sample, state):
    groups=list(dict.fromkeys(zip(sample,state)))
    lookup={g:i for i,g in enumerate(groups)}
    gi=np.fromiter((lookup[g] for g in zip(sample,state)),dtype=np.int64,count=len(sample))
    indicator=sparse.csr_matrix((np.ones(len(gi)),(gi,np.arange(len(gi)))),shape=(len(groups),len(gi)))
    n=np.asarray(indicator.sum(axis=1)).ravel()
    means=(indicator@x).toarray()/n[:,None]
    detected=x.copy(); detected.data=np.ones_like(detected.data)
    detection=(indicator@detected).toarray()/n[:,None]
    meta=pd.DataFrame(groups,columns=["sample","state"]); meta["n_cells"]=n.astype(int)
    return meta,means,detection


def adjusted_state_effects(meta, means, states):
    intercept=pd.DataFrame({"intercept":1.0},index=meta.index)
    sample_terms=pd.get_dummies(meta["sample"],prefix="sample",dtype=float).iloc[:,1:]
    state_terms=pd.get_dummies(pd.Categorical(meta["state"],categories=states),prefix="state",dtype=float).iloc[:,1:]
    design=pd.concat([intercept,sample_terms,state_terms],axis=1).to_numpy()
    coef=np.linalg.pinv(design)@means
    start=1+sample_terms.shape[1]
    effects=np.zeros((len(states),means.shape[1])); effects[1:]=coef[start:start+len(states)-1]
    return effects


def generate_panel_a():
    genes=[g for s in A_STATES for g in A_PROGRAMS[s]]
    adata=ad.read_h5ad(PANEL_A_H5AD,backed="r")
    obs=adata.obs[["status","sample","cell_subtype"]].copy()
    use=obs["status"].astype(str).eq("tumor") & obs["cell_subtype"].astype(str).isin(A_STATES)
    rows=np.flatnonzero(use.to_numpy())
    sample=obs.iloc[rows]["sample"].astype(str).to_numpy()
    state=obs.iloc[rows]["cell_subtype"].astype(str).to_numpy()
    missing=sorted(set(genes)-set(adata.raw.var_names.astype(str)))
    if missing: raise ValueError(f"Panel A genes absent from adata.raw: {missing}")
    cols=adata.raw.var_names.get_indexer(genes)
    x=adata.raw.X[rows,cols]
    if hasattr(x,"to_memory"): x=x.to_memory()
    x=sparse.csr_matrix(x,dtype=float); adata.file.close()
    meta,means,detection=aggregate_sample_state(x,sample,state)
    keep=meta["n_cells"].ge(MIN_A).to_numpy()
    meta=meta.loc[keep].reset_index(drop=True); means=means[keep]; detection=detection[keep]
    effects=adjusted_state_effects(meta,means,A_STATES)
    lo=effects.min(0); span=effects.max(0)-lo
    scaled=np.divide(effects-lo,span,out=np.zeros_like(effects),where=span>0)
    owner={g:s for s,gs in A_PROGRAMS.items() for g in gs}
    records=[]
    for si,s in enumerate(A_STATES):
        sm=meta["state"].eq(s).to_numpy()
        for gi,g in enumerate(genes):
            records.append({"state":s,"gene":g,"marker_program":owner[g],
                "sample_adjusted_state_effect":effects[si,gi],
                "per_gene_minmax_adjusted_expression":scaled[si,gi],
                "sample_balanced_mean_log_normalized_expression":float(means[sm,gi].mean()),
                "sample_balanced_detection_fraction":float(detection[sm,gi].mean()),
                "n_eligible_samples":int(sm.sum()),"n_tumor_cells":int(np.sum(state==s)),
                "minimum_cells_per_sample_state":MIN_A})
    return pd.DataFrame(records)


SOURCE_WORK = PROJECT_ROOT / "review_revision_20260828_li/results/tumor_epithelial_marker_publication_figure_repositioned/source_data_work"


def prepare_panel_b_inputs():
    states=["Epi_JUN","Epi_VIM"]
    adata=ad.read_h5ad(PANEL_B_H5AD,backed="r")
    obs=adata.obs[["status","sample","cell_subtype"]].copy()
    status=obs["status"].astype(str); sample=obs["sample"].astype(str); state=obs["cell_subtype"].astype(str)
    base=status.eq("tumor") & state.isin(states)
    tab=(pd.DataFrame({"sample":sample[base],"state":state[base]}).groupby(["sample","state"]).size().unstack(fill_value=0))
    paired=tab[(tab["Epi_JUN"]>=MIN_B)&(tab["Epi_VIM"]>=MIN_B)].index.astype(str).tolist()
    use=base & sample.isin(paired); rows=np.flatnonzero(use.to_numpy())
    sel_sample=sample.iloc[rows].to_numpy(); sel_state=state.iloc[rows].to_numpy()
    groups=[(s,st) for s in paired for st in states]; lookup={g:i for i,g in enumerate(groups)}
    gi=np.array([lookup[g] for g in zip(sel_sample,sel_state)])
    x=adata.X[rows,:]
    if hasattr(x,"to_memory"): x=x.to_memory()
    x=sparse.csr_matrix(x,dtype=float); genes=adata.var_names.astype(str).to_numpy(); adata.file.close()
    indicator=sparse.csr_matrix((np.ones(len(gi)),(gi,np.arange(len(gi)))),shape=(len(groups),len(gi)))
    counts=(indicator@x).toarray(); n=np.asarray(indicator.sum(1)).ravel()
    binary=x.copy(); binary.data=np.ones_like(binary.data); detection=(indicator@binary).toarray()/n[:,None]
    meta=pd.DataFrame({"library_id":[f"{s}__{st}" for s,st in groups],"sample":[s for s,_ in groups],"state":[st for _,st in groups],"n_cells":n.astype(int)})
    SOURCE_WORK.mkdir(parents=True,exist_ok=True)
    pd.DataFrame(counts.T,index=genes,columns=meta["library_id"]).round().astype(np.int64).to_csv(SOURCE_WORK/"panel_b_pseudobulk_counts.csv.gz",compression="gzip")
    pd.DataFrame(detection.T,index=genes,columns=meta["library_id"]).to_csv(SOURCE_WORK/"panel_b_detection_fraction.csv.gz",compression="gzip")
    meta.to_csv(SOURCE_WORK/"panel_b_pseudobulk_metadata.csv",index=False)
    return {"paired_tumors":len(paired),"selected_cells":len(rows),"libraries":len(meta)}


SOURCE_WORK.mkdir(parents=True,exist_ok=True)
publication_panel_a_source = generate_panel_a()
publication_panel_a_source.to_csv(SOURCE_WORK/"panel_a_source_data.csv",index=False)
panel_b_input_summary = prepare_panel_b_inputs()
print(f"Panel A: {len(publication_panel_a_source)} rows")
print(f"Panel B inputs: {panel_b_input_summary}")
print("Next run run_fig2_jun_vim_edger() from main_codes/fig2/fig2-2.R, then continue with the next notebook cell.")


In [ ]:
"""Assemble Panel B plotting source after running edgeR in fig2-2.R."""
from pathlib import Path
import numpy as np
import pandas as pd

counts_path=SOURCE_WORK/"panel_b_pseudobulk_counts.csv.gz"
detection_path=SOURCE_WORK/"panel_b_detection_fraction.csv.gz"
metadata_path=SOURCE_WORK/"panel_b_pseudobulk_metadata.csv"
edger_path=SOURCE_WORK/"panel_b_edger_results.csv.gz"
norm_path=SOURCE_WORK/"panel_b_edger_norm_factors.csv"
for required in (counts_path,detection_path,metadata_path,edger_path,norm_path):
    if not required.exists(): raise FileNotFoundError(f"Missing {required}; generate inputs in fig2-1 and run run_fig2_jun_vim_edger() in fig2-2.R")
counts=pd.read_csv(counts_path,index_col=0).T
detection=pd.read_csv(detection_path,index_col=0).T
meta=pd.read_csv(metadata_path)
counts=counts.loc[meta["library_id"]]; detection=detection.loc[meta["library_id"]]
norm=pd.read_csv(norm_path).set_index("library_id").loc[meta["library_id"]]
edger=pd.read_csv(edger_path).set_index("gene")
logcpm=np.log2(counts.to_numpy()/norm["effective_library_size"].to_numpy()[:,None]*1e6+1)
genes=counts.columns.astype(str).to_numpy(); gix={g:i for i,g in enumerate(genes)}
states=["Epi_JUN","Epi_VIM"]
owner={g:label for label,gs in B_GROUPS.items() for g in gs}
gene_order=[g for label in B_GROUPS for g in B_GROUPS[label] if g in gix]
records=[]
for g in gene_order:
    j=gix[g]; values={}
    for st in states:
        pos=meta["state"].eq(st).to_numpy(); values[st]=float(logcpm[pos,j].mean())
        records.append({"display_group":owner[g],"gene":g,"state":st,
            "mean_pseudobulk_log2_cpm":values[st],"mean_cell_detection_fraction":float(detection.loc[pos,g].mean()),
            "edgeR_log2FC_JUN_minus_VIM":float(edger.loc[g,"logFC"]) if g in edger.index else np.nan,
            "edgeR_PValue":float(edger.loc[g,"PValue"]) if g in edger.index else np.nan,
            "edgeR_FDR":float(edger.loc[g,"FDR"]) if g in edger.index else np.nan,
            "n_paired_tumors":int(meta["sample"].nunique()),"minimum_cells_per_state":MIN_B})
    center=np.mean(list(values.values()))
    for row in records[-2:]: row["centered_mean_log2_cpm"]=row["mean_pseudobulk_log2_cpm"]-center
publication_panel_b_source=pd.DataFrame(records)
publication_panel_b_source.to_csv(SOURCE_WORK/"panel_b_source_data.csv",index=False)
print(f"Panel B: {len(publication_panel_b_source)} rows from {meta['sample'].nunique()} paired tumors")


In [ ]:
"""Replot the tumor epithelial marker publication figure with local headers.

Panel A and Panel B numerical values are generated in the preceding notebook
cell from the original H5AD objects. Program lines and labels are placed above
the matching state-by-marker blocks.
"""

from __future__ import annotations

from collections import OrderedDict
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import font_manager
import numpy as np
import pandas as pd


PROJECT_ROOT = Path("/mnt/disk18t/lr_xcy/riku/kirc_205")
REVISION_ROOT = PROJECT_ROOT / "review_revision_20260828_li"
OUTDIR = REVISION_ROOT / "results/tumor_epithelial_marker_publication_figure_repositioned"

STATE_ORDER = [
    "Epi_MIOX",
    "Epi_GPX3",
    "Epi_ALDOB",
    "Epi_CA9",
    "Epi_JUN",
    "Epi_VIM",
    "Epi_AQP2",
    "Epi_CA12",
]

PANEL_A_PROGRAMS = OrderedDict(
    [
        ("Epi_MIOX", ["MIOX", "PCK1", "NAT8", "CUBN"]),
        ("Epi_GPX3", ["GPX3", "GATM", "ASS1", "PDZK1IP1"]),
        ("Epi_ALDOB", ["ALDOB", "FABP1", "APOE", "HPD"]),
        ("Epi_CA9", ["CA9", "NDUFA4L2", "EGLN3", "NDRG1"]),
        ("Epi_JUN", ["KLF6", "ZBTB20", "SOD2", "VMP1", "FOSB", "JUN"]),
        ("Epi_VIM", ["VIM", "LGALS1", "LGALS3", "NUPR1", "S100A6", "S100A4"]),
        ("Epi_AQP2", ["AQP2", "FXYD4", "HSD11B2", "DEFB1"]),
        ("Epi_CA12", ["CA12", "ATP6V0A4", "FOXI1", "ATP6V1G3"]),
    ]
)

PAIR_GROUP_ORDER = [
    "AP-1 / immediate-early",
    "JUN-associated, non-IEG",
    "VIM-associated",
]
PAIR_GROUP_OWNER = {
    "AP-1 / immediate-early": "Epi_JUN",
    "JUN-associated, non-IEG": "Epi_JUN",
    "VIM-associated": "Epi_VIM",
}

JUN_COLOR = "#D8892F"
VIM_COLOR = "#3F78A8"
NEUTRAL_COLOR = "#666666"
GRID_COLOR = "#E6E6E6"
TEXT_COLOR = "#252525"

BOX_HALF_HEIGHT = 0.41


def configure_style() -> None:
    regular = Path("/home/lr_xcy/.local/share/fonts/codex_trajectory_20260813/Arial.TTF")
    bold = Path("/home/lr_xcy/.local/share/fonts/codex_trajectory_20260813/Arialbd.TTF")
    for path in (regular, bold):
        if path.exists():
            font_manager.fontManager.addfont(path)
    family = "Arial" if regular.exists() else "DejaVu Sans"
    mpl.rcParams.update(
        {
            "font.family": family,
            "font.size": 8,
            "axes.titlesize": 10,
            "axes.labelsize": 8,
            "xtick.labelsize": 7,
            "ytick.labelsize": 8,
            "axes.linewidth": 0.7,
            "pdf.fonttype": 42,
            "ps.fonttype": 42,
            "svg.fonttype": "none",
            "savefig.facecolor": "white",
        }
    )


def style_axis(ax: mpl.axes.Axes) -> None:
    for spine in ax.spines.values():
        spine.set_color("#555555")
        spine.set_linewidth(0.65)
    ax.tick_params(width=0.65, length=2.5, color="#555555")


def ordered_panel_a_genes() -> tuple[list[str], list[tuple[str, int, int]]]:
    genes: list[str] = []
    spans: list[tuple[str, int, int]] = []
    for state, state_genes in PANEL_A_PROGRAMS.items():
        start = len(genes)
        genes.extend(state_genes)
        spans.append((state, start, len(genes) - 1))
    return genes, spans


def load_sources() -> tuple[pd.DataFrame, pd.DataFrame]:
    panel_a = publication_panel_a_source.copy()
    panel_b = publication_panel_b_source.copy()
    panel_a_genes, _ = ordered_panel_a_genes()
    missing_a = sorted(set(panel_a_genes) - set(panel_a["gene"]))
    if missing_a:
        raise ValueError(f"Panel A source data are missing genes: {missing_a}")
    missing_groups = sorted(set(PAIR_GROUP_ORDER) - set(panel_b["display_group"]))
    if missing_groups:
        raise ValueError(f"Panel B source data are missing groups: {missing_groups}")
    return panel_a, panel_b


def draw_box(
    ax: mpl.axes.Axes,
    start: int,
    end: int,
    row_y: float,
    color: str,
) -> None:
    ax.add_patch(
        mpl.patches.Rectangle(
            (start - 0.43, row_y - BOX_HALF_HEIGHT),
            end - start + 0.86,
            2 * BOX_HALF_HEIGHT,
            fill=False,
            edgecolor=color,
            linewidth=1.0,
            zorder=4,
        )
    )


def draw_headers_above_frame(
    ax: mpl.axes.Axes,
    headers: list[tuple[str, int, int, str]],
    line_y: float = 1.035,
    text_y: float = 1.065,
) -> None:
    """Draw all program lines and labels above the panel's outer frame."""
    transform = ax.get_xaxis_transform()
    for label, start, end, color in headers:
        ax.plot(
            [start - 0.36, end + 0.36],
            [line_y, line_y],
            transform=transform,
            color=color,
            linewidth=2.5,
            solid_capstyle="butt",
            clip_on=False,
            zorder=5,
        )
        ax.text(
            (start + end) / 2,
            text_y,
            label,
            transform=transform,
            ha="center",
            va="bottom",
            color=color,
            fontsize=7.0,
            fontweight="bold" if color in {JUN_COLOR, VIM_COLOR} else "normal",
            clip_on=False,
            zorder=6,
        )


def draw_panel_a(
    ax: mpl.axes.Axes,
    source: pd.DataFrame,
) -> mpl.collections.PathCollection:
    genes, spans = ordered_panel_a_genes()
    data = source[source["gene"].isin(genes)].copy()
    gene_x = {gene: index for index, gene in enumerate(genes)}
    state_y = {state: index for index, state in enumerate(STATE_ORDER)}

    ax.axhspan(
        state_y["Epi_JUN"] - 0.48,
        state_y["Epi_JUN"] + 0.48,
        color="#FFF5E7",
        zorder=0,
    )
    ax.axhspan(
        state_y["Epi_VIM"] - 0.48,
        state_y["Epi_VIM"] + 0.48,
        color="#EDF5FB",
        zorder=0,
    )

    detection = data["sample_balanced_detection_fraction"].to_numpy(dtype=float)
    sizes = 8 + 102 * np.power(detection, 1.35)
    scatter = ax.scatter(
        data["gene"].map(gene_x),
        data["state"].map(state_y),
        s=sizes,
        c=data["per_gene_minmax_adjusted_expression"],
        cmap="YlOrRd",
        vmin=0,
        vmax=1,
        edgecolor="#3B3B3B",
        linewidth=0.32,
        zorder=3,
    )

    for block_index, (state, start, end) in enumerate(spans):
        if block_index:
            ax.axvline(start - 0.5, color="#AAAAAA", linewidth=0.65, zorder=1)
        box_color = JUN_COLOR if state == "Epi_JUN" else VIM_COLOR if state == "Epi_VIM" else "#777777"
        draw_box(ax, start, end, state_y[state], box_color)

    draw_headers_above_frame(
        ax,
        [
            (
                state.replace("Epi_", ""),
                start,
                end,
                JUN_COLOR if state == "Epi_JUN" else VIM_COLOR if state == "Epi_VIM" else NEUTRAL_COLOR,
            )
            for state, start, end in spans
        ],
    )

    ax.set_xlim(-0.65, len(genes) - 0.35)
    ax.set_ylim(len(STATE_ORDER) - 0.45, -0.48)
    ax.set_xticks(range(len(genes)))
    ax.set_xticklabels(genes, rotation=58, ha="right", rotation_mode="anchor", fontsize=6.4)
    ax.set_yticks([state_y[state] for state in STATE_ORDER])
    ax.set_yticklabels(STATE_ORDER, fontsize=7.7)
    for tick in ax.get_yticklabels():
        if tick.get_text() == "Epi_JUN":
            tick.set_color(JUN_COLOR)
            tick.set_fontweight("bold")
        elif tick.get_text() == "Epi_VIM":
            tick.set_color(VIM_COLOR)
            tick.set_fontweight("bold")
    ax.grid(color=GRID_COLOR, linewidth=0.48)
    ax.set_axisbelow(True)
    style_axis(ax)

    ax.text(-0.03, 1.44, "A", transform=ax.transAxes, fontsize=12, fontweight="bold", va="top")
    ax.text(
        0.015,
        1.44,
        "Tumor-only epithelial marker landscape",
        transform=ax.transAxes,
        fontsize=10,
        fontweight="bold",
        color=TEXT_COLOR,
        va="top",
    )
    ax.text(
        0.015,
        1.30,
        "46,038 epithelial cells from 61 tumors; sample-adjusted expression (sample + state model)",
        transform=ax.transAxes,
        fontsize=7.2,
        color="#555555",
        va="top",
    )
    return scatter


def draw_panel_b(
    ax: mpl.axes.Axes,
    effect_ax: mpl.axes.Axes,
    source: pd.DataFrame,
) -> tuple[mpl.collections.PathCollection, mpl.cm.ScalarMappable]:
    genes = source["gene"].drop_duplicates().tolist()
    gene_x = {gene: index for index, gene in enumerate(genes)}
    states = ["Epi_JUN", "Epi_VIM"]
    state_y = {state: index for index, state in enumerate(states)}

    values = source["centered_mean_log2_cpm"].to_numpy(dtype=float)
    cap = max(0.35, float(np.nanquantile(np.abs(values), 0.95)))
    cap = min(cap, 0.75)
    norm = mpl.colors.TwoSlopeNorm(vmin=-cap, vcenter=0, vmax=cap)
    detection = source["mean_cell_detection_fraction"].to_numpy(dtype=float)
    sizes = 8 + 118 * np.power(detection, 1.35)
    scatter = ax.scatter(
        source["gene"].map(gene_x),
        source["state"].map(state_y),
        s=sizes,
        c=values,
        cmap="RdBu_r",
        norm=norm,
        edgecolor="#363636",
        linewidth=0.34,
        zorder=3,
    )

    ax.axhspan(state_y["Epi_JUN"] - 0.46, state_y["Epi_JUN"] + 0.46, color="#FFF5E7", zorder=0)
    ax.axhspan(state_y["Epi_VIM"] - 0.46, state_y["Epi_VIM"] + 0.46, color="#EDF5FB", zorder=0)

    spans: list[tuple[str, int, int]] = []
    for group in PAIR_GROUP_ORDER:
        group_genes = source.loc[source["display_group"].eq(group), "gene"].drop_duplicates().tolist()
        positions = [gene_x[gene] for gene in group_genes]
        spans.append((group, min(positions), max(positions)))

    for group_index, (group, start, end) in enumerate(spans):
        if group_index:
            ax.axvline(start - 0.5, color="#AAAAAA", linewidth=0.65, zorder=1)

    draw_headers_above_frame(
        ax,
        [
            (
                group,
                start,
                end,
                JUN_COLOR if PAIR_GROUP_OWNER[group] == "Epi_JUN" else VIM_COLOR,
            )
            for group, start, end in spans
        ],
        line_y=1.055,
        text_y=1.095,
    )

    ax.set_xlim(-0.65, len(genes) - 0.35)
    ax.set_ylim(1.48, -0.48)
    ax.set_xticks(range(len(genes)))
    ax.set_xticklabels([])
    ax.tick_params(axis="x", which="both", bottom=False, labelbottom=False)
    ax.set_yticks([state_y[state] for state in states])
    ax.set_yticklabels(states, fontsize=7.7)
    ax.get_yticklabels()[0].set_color(JUN_COLOR)
    ax.get_yticklabels()[0].set_fontweight("bold")
    ax.get_yticklabels()[1].set_color(VIM_COLOR)
    ax.get_yticklabels()[1].set_fontweight("bold")
    ax.grid(color=GRID_COLOR, linewidth=0.48)
    ax.set_axisbelow(True)
    style_axis(ax)

    ax.text(-0.03, 1.48, "B", transform=ax.transAxes, fontsize=12, fontweight="bold", va="top")
    ax.text(
        0.015,
        1.48,
        "Paired Epi_JUN–Epi_VIM contrast",
        transform=ax.transAxes,
        fontsize=10,
        fontweight="bold",
        color=TEXT_COLOR,
        va="top",
    )
    ax.text(
        0.015,
        1.30,
        "41 tumors with ≥20 cells in each state; color shows expression relative to the gene mean",
        transform=ax.transAxes,
        fontsize=7.2,
        color="#555555",
        va="top",
    )

    stats = source.drop_duplicates("gene").set_index("gene").loc[genes]
    logfc = stats["edgeR_log2FC_JUN_minus_VIM"].to_numpy(dtype=float)
    fdr = stats["edgeR_FDR"].to_numpy(dtype=float)
    effect_cap = max(0.5, float(np.nanquantile(np.abs(logfc), 0.95)))
    effect_cap = min(effect_cap, 1.25)
    effect_norm = mpl.colors.TwoSlopeNorm(vmin=-effect_cap, vcenter=0, vmax=effect_cap)
    effect_map = mpl.cm.ScalarMappable(norm=effect_norm, cmap="RdBu_r")
    effect_ax.imshow(
        logfc[np.newaxis, :],
        cmap="RdBu_r",
        norm=effect_norm,
        aspect="auto",
        interpolation="nearest",
        extent=(-0.5, len(genes) - 0.5, -0.5, 0.5),
    )
    effect_ax.set_xlim(-0.65, len(genes) - 0.35)
    effect_ax.set_ylim(-0.5, 0.5)
    effect_ax.set_yticks([0])
    effect_ax.set_yticklabels(["edgeR\nlog2FC"], fontsize=6.7)
    effect_ax.set_xticks(range(len(genes)))
    effect_ax.set_xticklabels(genes, rotation=58, ha="right", rotation_mode="anchor", fontsize=6.4)
    effect_ax.set_xticks(np.arange(-0.5, len(genes), 1), minor=True)
    effect_ax.grid(which="minor", axis="x", color="white", linewidth=0.7)
    effect_ax.tick_params(which="minor", bottom=False)
    effect_ax.tick_params(axis="x", pad=2)
    for x, qvalue in enumerate(fdr):
        if np.isfinite(qvalue) and qvalue < 0.05:
            effect_ax.text(x, 0, "•", ha="center", va="center", fontsize=6.5, color="white", fontweight="bold")
    style_axis(effect_ax)
    return scatter, effect_map


def add_legends(
    fig: mpl.figure.Figure,
    panel_a_scatter: mpl.collections.PathCollection,
    panel_b_scatter: mpl.collections.PathCollection,
    effect_map: mpl.cm.ScalarMappable,
    panel_a_ax: mpl.axes.Axes,
    panel_b_ax: mpl.axes.Axes,
) -> None:
    cax_a = fig.add_axes([0.925, 0.655, 0.0105, 0.145])
    cb_a = fig.colorbar(panel_a_scatter, cax=cax_a)
    cb_a.set_ticks([0, 0.5, 1])
    cb_a.ax.tick_params(labelsize=6.5, width=0.5, length=2)
    cb_a.set_label("Adjusted expression\nscaled within gene", fontsize=6.8, labelpad=5)

    cax_b = fig.add_axes([0.925, 0.265, 0.0105, 0.105])
    cb_b = fig.colorbar(panel_b_scatter, cax=cax_b)
    cb_b.ax.tick_params(labelsize=6.5, width=0.5, length=2)
    cb_b.set_label("Centered mean\nlog2(CPM + 1)", fontsize=6.8, labelpad=5)

    cax_effect = fig.add_axes([0.925, 0.095, 0.0105, 0.075])
    cb_effect = fig.colorbar(effect_map, cax=cax_effect)
    cb_effect.ax.tick_params(labelsize=6.5, width=0.5, length=2)
    cb_effect.set_label("log2FC\nJUN − VIM", fontsize=6.8, labelpad=5)

    fractions = [0.25, 0.50, 0.75, 1.00]
    handles = [
        panel_a_ax.scatter(
            [],
            [],
            s=8 + 102 * fraction**1.35,
            facecolor="#CB5B3F",
            edgecolor="#363636",
            linewidth=0.32,
        )
        for fraction in fractions
    ]
    fig.legend(
        handles,
        [f"{int(fraction * 100)}%" for fraction in fractions],
        title="Cells detected",
        frameon=False,
        loc="upper right",
        bbox_to_anchor=(0.900, 1.005),
        borderaxespad=0,
        ncol=4,
        columnspacing=0.9,
        handletextpad=0.55,
        fontsize=6.3,
        title_fontsize=6.7,
    )


def main() -> None:
    configure_style()
    OUTDIR.mkdir(parents=True, exist_ok=True)
    panel_a, panel_b = load_sources()

    fig = plt.figure(figsize=(11.8, 7.5), facecolor="white")
    grid = fig.add_gridspec(
        nrows=4,
        ncols=1,
        # Give Panel A and Panel B dotplots exactly the same vertical extent.
        # The edgeR strip retains its original height.
        height_ratios=[1.50, 0.35, 1.50, 0.38],
        left=0.085,
        right=0.900,
        bottom=0.09,
        top=0.925,
        hspace=0.88,
    )
    ax_a = fig.add_subplot(grid[0, 0])
    ax_b = fig.add_subplot(grid[2, 0])
    ax_effect = fig.add_subplot(grid[3, 0])

    scatter_a = draw_panel_a(ax_a, panel_a)
    scatter_b, effect_map = draw_panel_b(ax_b, ax_effect, panel_b)
    add_legends(fig, scatter_a, scatter_b, effect_map, ax_a, ax_b)

    fig.text(
        0.085,
        0.018,
        "Panel A uses sample-adjusted tumor expression; Panel B uses paired tumor edgeR. Program labels and lines are positioned above each dotplot's outer frame. "
        "White dots in the edgeR strip denote FDR < 0.05.",
        ha="left",
        va="bottom",
        fontsize=6.5,
        color="#555555",
    )

    base = OUTDIR / "tumor_epithelial_marker_publication_figure_headers_above_panel_frames_equal_y_title_clearance"
    fig.savefig(base.with_suffix(".pdf"), bbox_inches="tight")
    plt.close(fig)
    print(f"output={base.with_suffix('.pdf')}")

if __name__ == "__main__":
    main()


## Six-state epithelial and mesenchymal/EMT-associated marker dotplot

Output: `tumor_epi6_sample_adjusted_epithelial_emt_dotplot_jun_last_compact_xy_headers_above_frame.pdf`.


In [ ]:
"""Draw a tumor-only, sample-adjusted epithelial/EMT marker dotplot.

The analysis follows the Panel A method used in the 2026-08-19 reviewer
revision: aggregate log-normalized expression by sample and epithelial state,
retain sample-state combinations with at least 10 cells, fit an additive
sample + state model, and min-max scale the fitted state effects within each
gene. Dot size is the equal-sample mean detection fraction. The source H5AD
and the 2026-08-19 revision directory are read only.
"""

from __future__ import annotations

from pathlib import Path

import anndata as ad
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import font_manager
import numpy as np
import pandas as pd
from scipy import sparse


MIN_CELLS_PER_SAMPLE_STATE = 10

PROJECT_ROOT = Path("/mnt/disk18t/lr_xcy/riku/kirc_205")
REVISION_ROOT = PROJECT_ROOT / "review_revision_20260828_li"
INPUT_H5AD = PROJECT_ROOT / "leiden_detailed/adata_epi.h5ad"
OUTDIR = REVISION_ROOT / "results/tumor_epi6_emt_marker_dotplot"

STATE_ORDER = [
    "Epi_MIOX",
    "Epi_GPX3",
    "Epi_ALDOB",
    "Epi_CA9",
    "Epi_VIM",
    "Epi_JUN",
]

MARKER_GROUPS = {
    "Epithelial markers": [
        "CDH1",
        "OCLN",
        "CLDN1",
        "CLDN4",
        "EPCAM",
        "KRT8",
        "KRT18",
        "MUC1",
    ],
    "Mesenchymal / EMT-associated markers": [
        "VIM",
        "CDH2",
        "SNAI1",
        "SNAI2",
        "TWIST1",
        "TWIST2",
        "ZEB1",
        "ZEB2",
        "FN1",
        "ACTA2",
        "MMP2",
        "MMP9",
    ],
}

JUN_COLOR = "#D8892F"
VIM_COLOR = "#3F78A8"
GRID_COLOR = "#E6E6E6"
TEXT_COLOR = "#252525"


def configure_style() -> None:
    regular = Path("/home/lr_xcy/.local/share/fonts/codex_trajectory_20260813/Arial.TTF")
    bold = Path("/home/lr_xcy/.local/share/fonts/codex_trajectory_20260813/Arialbd.TTF")
    for path in (regular, bold):
        if path.exists():
            font_manager.fontManager.addfont(path)
    family = "Arial" if regular.exists() else "DejaVu Sans"
    mpl.rcParams.update(
        {
            "font.family": family,
            "font.size": 8,
            "axes.titlesize": 10,
            "axes.labelsize": 8,
            "xtick.labelsize": 7,
            "ytick.labelsize": 8,
            "axes.linewidth": 0.7,
            "pdf.fonttype": 42,
            "ps.fonttype": 42,
            "svg.fonttype": "none",
            "savefig.facecolor": "white",
        }
    )


def ordered_markers() -> tuple[list[str], list[tuple[str, int, int]]]:
    genes: list[str] = []
    spans: list[tuple[str, int, int]] = []
    for label, group_genes in MARKER_GROUPS.items():
        start = len(genes)
        genes.extend(group_genes)
        spans.append((label, start, len(genes) - 1))
    return genes, spans


def aggregate_sample_state(
    matrix: sparse.csr_matrix,
    sample: np.ndarray,
    state: np.ndarray,
) -> tuple[pd.DataFrame, np.ndarray, np.ndarray]:
    groups = list(dict.fromkeys(zip(sample, state)))
    lookup = {group: idx for idx, group in enumerate(groups)}
    group_index = np.fromiter(
        (lookup[group] for group in zip(sample, state)),
        dtype=np.int64,
        count=len(sample),
    )
    indicator = sparse.csr_matrix(
        (
            np.ones(len(group_index), dtype=np.float64),
            (group_index, np.arange(len(group_index))),
        ),
        shape=(len(groups), len(group_index)),
    )
    n_cells = np.asarray(indicator.sum(axis=1)).ravel()
    sums = (indicator @ matrix).toarray()
    means = sums / n_cells[:, None]

    detected = matrix.copy()
    detected.data = np.ones_like(detected.data)
    detection_fraction = (indicator @ detected).toarray() / n_cells[:, None]

    metadata = pd.DataFrame(groups, columns=["sample", "state"])
    metadata["n_cells"] = n_cells.astype(int)
    return metadata, means, detection_fraction


def sample_adjusted_state_effects(
    metadata: pd.DataFrame,
    group_means: np.ndarray,
) -> np.ndarray:
    """Fit expression ~ sample + state for all displayed genes."""
    intercept = pd.DataFrame({"intercept": 1.0}, index=metadata.index)
    sample_terms = pd.get_dummies(metadata["sample"], prefix="sample", dtype=float).iloc[:, 1:]
    state_terms = pd.get_dummies(
        pd.Categorical(metadata["state"], categories=STATE_ORDER),
        prefix="state",
        dtype=float,
    ).iloc[:, 1:]
    design = pd.concat([intercept, sample_terms, state_terms], axis=1).to_numpy()
    if np.linalg.matrix_rank(design) != design.shape[1]:
        raise ValueError("The sample + state design matrix is not full rank")
    coefficients = np.linalg.pinv(design) @ group_means

    state_start = 1 + sample_terms.shape[1]
    effects = np.zeros((len(STATE_ORDER), group_means.shape[1]), dtype=float)
    effects[1:, :] = coefficients[state_start : state_start + len(STATE_ORDER) - 1, :]
    return effects


def make_source_data() -> tuple[pd.DataFrame, pd.DataFrame, dict[str, int]]:
    marker_order, _ = ordered_markers()
    adata = ad.read_h5ad(INPUT_H5AD, backed="r")
    if adata.raw is None:
        raise ValueError("adata.raw is absent; expected full-gene log-normalized expression")

    available = set(adata.raw.var_names.astype(str))
    missing = [gene for gene in marker_order if gene not in available]
    if missing:
        raise ValueError(f"Markers absent from adata.raw: {missing}")

    obs = adata.obs[["status", "sample", "cell_subtype"]].copy()
    status = obs["status"].astype(str)
    sample = obs["sample"].astype(str)
    state = obs["cell_subtype"].astype(str)
    use = status.eq("tumor") & state.isin(STATE_ORDER)
    cell_indices = np.flatnonzero(use.to_numpy())
    selected_sample = sample.iloc[cell_indices].to_numpy()
    selected_state = state.iloc[cell_indices].to_numpy()

    gene_indices = adata.raw.var_names.get_indexer(marker_order)
    matrix = adata.raw.X[cell_indices, gene_indices]
    if hasattr(matrix, "to_memory"):
        matrix = matrix.to_memory()
    matrix = sparse.csr_matrix(matrix, dtype=np.float64)
    adata.file.close()

    metadata, group_means, group_detection = aggregate_sample_state(
        matrix, selected_sample, selected_state
    )
    metadata["eligible"] = metadata["n_cells"].ge(MIN_CELLS_PER_SAMPLE_STATE)
    eligible = metadata["eligible"].to_numpy()
    eligible_metadata = metadata.loc[eligible].reset_index(drop=True)
    eligible_means = group_means[eligible, :]
    eligible_detection = group_detection[eligible, :]

    missing_states = sorted(set(STATE_ORDER) - set(eligible_metadata["state"]))
    if missing_states:
        raise ValueError(f"No eligible sample-state groups for: {missing_states}")

    effects = sample_adjusted_state_effects(eligible_metadata, eligible_means)
    minima = effects.min(axis=0)
    maxima = effects.max(axis=0)
    ranges = maxima - minima
    scaled = np.divide(
        effects - minima,
        ranges,
        out=np.zeros_like(effects),
        where=ranges > 0,
    )

    gene_to_group = {
        gene: label for label, genes in MARKER_GROUPS.items() for gene in genes
    }
    records: list[dict[str, object]] = []
    for state_i, epithelial_state in enumerate(STATE_ORDER):
        state_mask = eligible_metadata["state"].eq(epithelial_state).to_numpy()
        for gene_i, gene in enumerate(marker_order):
            records.append(
                {
                    "state": epithelial_state,
                    "gene": gene,
                    "marker_group": gene_to_group[gene],
                    "sample_adjusted_state_effect": effects[state_i, gene_i],
                    "per_gene_minmax_adjusted_expression": scaled[state_i, gene_i],
                    "sample_balanced_mean_log_normalized_expression": float(
                        eligible_means[state_mask, gene_i].mean()
                    ),
                    "sample_balanced_detection_fraction": float(
                        eligible_detection[state_mask, gene_i].mean()
                    ),
                    "n_eligible_samples": int(state_mask.sum()),
                    "n_tumor_cells": int(np.sum(selected_state == epithelial_state)),
                    "minimum_cells_per_sample_state": MIN_CELLS_PER_SAMPLE_STATE,
                }
            )

    summary = {
        "tumor_cells": int(len(cell_indices)),
        "tumor_samples": int(pd.Series(selected_sample).nunique()),
        "eligible_sample_state_groups": int(eligible.sum()),
    }
    return pd.DataFrame(records), metadata, summary


def add_group_headers(
    ax: mpl.axes.Axes,
    spans: list[tuple[str, int, int]],
) -> None:
    colors = ["#555555", "#3F78A8"]
    for idx, (label, start, end) in enumerate(spans):
        if idx > 0:
            ax.axvline(start - 0.5, color="#8F8F8F", linewidth=0.8, zorder=1)
        ax.plot(
            [start - 0.36, end + 0.36],
            [1.045, 1.045],
            transform=ax.get_xaxis_transform(),
            color=colors[idx],
            linewidth=2.8,
            solid_capstyle="butt",
            clip_on=False,
        )
        ax.text(
            (start + end) / 2,
            1.075,
            label,
            transform=ax.get_xaxis_transform(),
            ha="center",
            va="bottom",
            color=colors[idx],
            fontsize=8,
            fontweight="bold",
            clip_on=False,
        )


def draw_dotplot(source: pd.DataFrame, summary: dict[str, int]) -> None:
    marker_order, spans = ordered_markers()
    gene_x = {gene: idx for idx, gene in enumerate(marker_order)}
    state_y = {state: idx for idx, state in enumerate(STATE_ORDER)}

    fig, ax = plt.subplots(figsize=(10.6, 5.5), facecolor="white")
    ax.axhspan(state_y["Epi_JUN"] - 0.48, state_y["Epi_JUN"] + 0.48, color="#FFF5E7", zorder=0)
    ax.axhspan(state_y["Epi_VIM"] - 0.48, state_y["Epi_VIM"] + 0.48, color="#EDF5FB", zorder=0)

    detection = source["sample_balanced_detection_fraction"].to_numpy(dtype=float)
    sizes = 12 + 145 * np.power(detection, 1.35)
    scatter = ax.scatter(
        source["gene"].map(gene_x),
        source["state"].map(state_y),
        s=sizes,
        c=source["per_gene_minmax_adjusted_expression"],
        cmap="YlOrRd",
        vmin=0,
        vmax=1,
        edgecolor="#3B3B3B",
        linewidth=0.36,
        zorder=3,
    )

    ax.set_xlim(-0.65, len(marker_order) - 0.35)
    ax.set_ylim(len(STATE_ORDER) - 0.45, -0.48)
    ax.set_xticks(range(len(marker_order)))
    ax.set_xticklabels(marker_order, rotation=58, ha="right", rotation_mode="anchor", fontsize=7.5)
    ax.set_yticks(range(len(STATE_ORDER)))
    ax.set_yticklabels(STATE_ORDER, fontsize=8.8)
    for tick in ax.get_yticklabels():
        if tick.get_text() == "Epi_JUN":
            tick.set_color(JUN_COLOR)
            tick.set_fontweight("bold")
        elif tick.get_text() == "Epi_VIM":
            tick.set_color(VIM_COLOR)
            tick.set_fontweight("bold")

    ax.grid(color=GRID_COLOR, linewidth=0.52)
    ax.set_axisbelow(True)
    for spine in ax.spines.values():
        spine.set_color("#555555")
        spine.set_linewidth(0.7)
    ax.tick_params(width=0.65, length=2.5, color="#555555")
    add_group_headers(ax, spans)

    ax.text(
        0,
        1.70,
        "Tumor-only epithelial and EMT marker landscape",
        transform=ax.transAxes,
        fontsize=11,
        fontweight="bold",
        color=TEXT_COLOR,
        va="top",
    )
    ax.text(
        0,
        1.45,
        f"{summary['tumor_cells']:,} epithelial cells from {summary['tumor_samples']} tumors; "
        "sample-adjusted expression (sample + state model)",
        transform=ax.transAxes,
        fontsize=7.8,
        color="#555555",
        va="top",
    )

    color_ax = fig.add_axes([0.895, 0.36, 0.012, 0.31])
    colorbar = fig.colorbar(scatter, cax=color_ax)
    colorbar.set_ticks([0, 0.5, 1])
    colorbar.ax.tick_params(labelsize=7, width=0.5, length=2)
    colorbar.set_label("Adjusted expression\nscaled within gene", fontsize=7.4, labelpad=5)

    fractions = [0.25, 0.50, 0.75, 1.00]
    handles = [
        ax.scatter(
            [],
            [],
            s=12 + 145 * fraction**1.35,
            facecolor="#CB5B3F",
            edgecolor="#363636",
            linewidth=0.36,
        )
        for fraction in fractions
    ]
    fig.legend(
        handles,
        [f"{int(fraction * 100)}%" for fraction in fractions],
        title="Cells detected",
        frameon=False,
        loc="upper right",
        bbox_to_anchor=(0.875, 0.955),
        borderaxespad=0,
        ncol=4,
        columnspacing=1.0,
        handletextpad=0.55,
        fontsize=6.8,
        title_fontsize=7.2,
    )

    fig.text(
        0.085,
        0.025,
        "Tumor-only visualization. Each eligible sample-state combination contributes one equally weighted mean; "
        "dot size is the equal-sample mean detection fraction.",
        ha="left",
        va="bottom",
        fontsize=6.7,
        color="#555555",
    )
    # Compress only the dotplot in the vertical direction while retaining
    # the original canvas width and clear title/label margins.
    fig.subplots_adjust(left=0.10, right=0.86, bottom=0.33, top=0.63)
    base = OUTDIR / "tumor_epi6_sample_adjusted_epithelial_emt_dotplot_jun_last_compact_xy_headers_above_frame"
    fig.savefig(base.with_suffix(".pdf"), bbox_inches="tight")
    plt.close(fig)


def main() -> None:
    configure_style()
    OUTDIR.mkdir(parents=True, exist_ok=True)
    source, _, summary = make_source_data()
    draw_dotplot(source, summary)
    output = OUTDIR / "tumor_epi6_sample_adjusted_epithelial_emt_dotplot_jun_last_compact_xy_headers_above_frame.pdf"
    print(f"output={output}")

if __name__ == "__main__":
    main()


## Three-state epithelial marker programs


In [ ]:
"""Generate source data and draw the three-state marker-program panel."""
from pathlib import Path
import anndata as ad
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.patches import Rectangle
import numpy as np
import pandas as pd
from scipy import sparse

ROOT=Path("/mnt/disk18t/lr_xcy/riku/kirc_205")
INPUT=ROOT/"leiden_detailed/adata_epi.h5ad"
OUT=ROOT/"review_revision_20260828_li/results/figure2a_state_marker_programs_matched_theme"
STATES=["Epi_CA9","Epi_VIM","Epi_JUN"]
COLORS={"Epi_CA9":"#555555","Epi_VIM":"#3F78A8","Epi_JUN":"#D8892F"}
GENES=["CA9","EGLN3","NDRG1","VIM","S100A6","S100A4","PDK4","NLGN1","ZBTB20"]
GROUP={g:("CA9" if i<3 else "VIM" if i<6 else "JUN") for i,g in enumerate(GENES)}
BG={"CA9":"#F5F5F5","VIM":"#EDF5FB","JUN":"#FFF5E7"}

adata=ad.read_h5ad(INPUT,backed="r")
obs=adata.obs[["status","cell_subtype"]].copy(); use=obs["status"].astype(str).eq("tumor") & obs["cell_subtype"].astype(str).isin(STATES)
rows=np.flatnonzero(use.to_numpy()); states=obs.iloc[rows]["cell_subtype"].astype(str).to_numpy()
cols=adata.raw.var_names.get_indexer(GENES); x=adata.raw.X[rows,cols]
if hasattr(x,"to_memory"): x=x.to_memory()
x=sparse.csr_matrix(x,dtype=float); adata.file.close()
records=[]
for st in STATES:
    sm=states==st; mean=np.asarray(x[sm].mean(0)).ravel(); det=np.asarray((x[sm]>0).mean(0)).ravel()
    for i,g in enumerate(GENES): records.append({"state":st,"gene":g,"marker_group":GROUP[g],"mean_log_normalized_expression":mean[i],"detection_fraction":det[i]})
figure2a_source=pd.DataFrame(records)
figure2a_source["triad_gene_minmax_expression"]=figure2a_source.groupby("gene")["mean_log_normalized_expression"].transform(lambda v:(v-v.min())/(v.max()-v.min()) if v.max()>v.min() else 0)

regular=Path("/home/lr_xcy/.local/share/fonts/codex_trajectory_20260813/Arial.TTF"); bold=regular.with_name("Arialbd.TTF")
for f in (regular,bold):
    if f.exists(): font_manager.fontManager.addfont(f)
mpl.rcParams.update({"font.family":"Arial" if regular.exists() else "DejaVu Sans","font.size":8,"axes.linewidth":.7,"pdf.fonttype":42,"ps.fonttype":42,"savefig.facecolor":"white"})
fig,ax=plt.subplots(figsize=(6.4,6.2),facecolor="white"); fig.subplots_adjust(left=.22,right=.86,bottom=.14,top=.84)
sx={s:i for i,s in enumerate(STATES)}; gy={g:i for i,g in enumerate(GENES)}
for i,g in enumerate(GENES): ax.add_patch(Rectangle((-.48,i-.47),2.96,.94,facecolor=BG[GROUP[g]],edgecolor="none",zorder=0))
sc=ax.scatter(figure2a_source["state"].map(sx),figure2a_source["gene"].map(gy),s=38+520*figure2a_source["detection_fraction"],c=figure2a_source["triad_gene_minmax_expression"],cmap="YlOrRd",vmin=0,vmax=1,edgecolor="#3B3B3B",linewidth=.55,zorder=3)
ax.set(xlim=(-.55,2.55),ylim=(len(GENES)-.45,-.55)); ax.set_xticks(range(3),STATES,fontweight="bold",fontsize=11); ax.xaxis.tick_top(); ax.tick_params(axis="x",length=3,width=.7,color="#555555",pad=9)
for lab,st in zip(ax.get_xticklabels(),STATES): lab.set_color(COLORS[st])
ax.set_yticks(range(len(GENES)),GENES,fontsize=9); ax.tick_params(axis="y",length=3,width=.7,color="#555555",pad=7)
for b in [2.5,5.5]: ax.axhline(b,color="white",lw=5,zorder=2)
ax.grid(color="#E6E6E6",lw=.55); ax.set_axisbelow(True)
for sp in ax.spines.values(): sp.set_visible(True); sp.set_color("#555555"); sp.set_linewidth(.7)
fig.text(.045,.99,"A",ha="left",va="top",fontsize=18,fontweight="bold",color="#252525")
fig.text(.15,.99,"Tumour epithelial states retain distinct marker programs",ha="left",va="top",fontsize=14,fontweight="bold",color="#252525")
fig.text(.15,.925,"Tumour cells only; colour is scaled within each gene",ha="left",va="top",fontsize=10,color="#555555")
fracs=[.25,.75]; handles=[ax.scatter([],[],s=38+520*f,facecolor="#CB5B3F",edgecolor="#3B3B3B",linewidth=.55) for f in fracs]
ax.legend(handles,[f"{int(f*100)}% detected" for f in fracs],title="Dot size",frameon=False,loc="lower left",bbox_to_anchor=(1.08,0),borderaxespad=0,labelspacing=1.5,handletextpad=1,fontsize=8,title_fontsize=8.5)
cax=fig.add_axes([.91,.38,.018,.18]); cb=fig.colorbar(sc,cax=cax); cb.set_ticks([0,.5,1]); cb.ax.tick_params(labelsize=7,width=.5,length=2); cb.set_label("Relative expression\nscaled within gene",fontsize=7.5,labelpad=5)
OUT.mkdir(parents=True,exist_ok=True); target=OUT/"Figure2A_state_marker_programs_matched_theme_epi_labels_xy_ticks_with_color_legend.pdf"; fig.savefig(target,bbox_inches="tight"); plt.close(fig); print(target)


## CA9–VIM–JUN epithelial bridge


In [ ]:
"""Generate the CA9-to-JUN PCA projection source and draw the bridge UMAP."""
from pathlib import Path
import anndata as ad
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import FancyArrowPatch
import numpy as np
import pandas as pd

ROOT=Path("/mnt/disk18t/lr_xcy/riku/kirc_205"); INPUT=ROOT/"leiden_detailed/adata_epi.h5ad"
OUT=ROOT/"review_revision_20260828_li/results/figure2b_epithelial_bridge_umap_larger_labels"
STATES=["Epi_CA9","Epi_VIM","Epi_JUN"]; COLORS={"Epi_CA9":"#6B5CA5","Epi_VIM":"#2A9D8F","Epi_JUN":"#E67E22"}; LABEL={s:s.replace("Epi_","") for s in STATES}
adata=ad.read_h5ad(INPUT,backed="r"); obs=adata.obs[["status","cell_subtype"]].copy(); use=obs["status"].astype(str).eq("tumor") & obs["cell_subtype"].astype(str).isin(STATES); rows=np.flatnonzero(use.to_numpy())
state=obs.iloc[rows]["cell_subtype"].astype(str).to_numpy(); pca=np.asarray(adata.obsm["X_pca_inte"][rows,:30]); umap=np.asarray(adata.obsm["X_umap"][rows,:2]); adata.file.close()
ca9=pca[state=="Epi_CA9"].mean(0); jun=pca[state=="Epi_JUN"].mean(0); direction=jun-ca9; coordinate=((pca-ca9)@direction)/float(direction@direction)
figure2b_source=pd.DataFrame({"state":state,"umap_1":umap[:,0],"umap_2":umap[:,1],"continuum_coordinate":coordinate})
regular=Path("/home/lr_xcy/.local/share/fonts/codex_trajectory_20260813/Arial.TTF"); bold=regular.with_name("Arialbd.TTF")
for f in (regular,bold):
    if f.exists(): font_manager.fontManager.addfont(f)
mpl.rcParams.update({"font.family":"Arial" if regular.exists() else "DejaVu Sans","font.size":8,"pdf.fonttype":42,"ps.fonttype":42,"savefig.facecolor":"white"})
fig,ax=plt.subplots(figsize=(7.6,6.2)); fig.subplots_adjust(left=.08,right=.91,bottom=.08,top=.84)
fig.text(.045,.99,"B",ha="left",va="top",fontsize=18,fontweight="bold",color="#263238"); fig.text(.15,.99,"VIM occupies the bridge between CA9 and JUN",ha="left",va="top",fontsize=14,fontweight="bold",color="#263238"); fig.text(.15,.925,f"{len(figure2b_source):,} tumour epithelial cells; UMAP is used for visualization",ha="left",va="top",fontsize=10,color="#66717B")
cmap=LinearSegmentedColormap.from_list("continuum",[COLORS["Epi_CA9"],"#6FB6B1",COLORS["Epi_JUN"]]); low,high=figure2b_source["continuum_coordinate"].quantile([.01,.99]); pts=ax.scatter(figure2b_source.umap_1,figure2b_source.umap_2,c=figure2b_source.continuum_coordinate,cmap=cmap,vmin=low,vmax=high,s=2,alpha=.42,lw=0,rasterized=True)
cent=figure2b_source.groupby("state")[["umap_1","umap_2"]].median().loc[STATES]
for st in STATES:
    x,y=cent.loc[st]; ax.scatter(x,y,s=430,facecolor="white",edgecolor=COLORS[st],linewidth=2.4,zorder=6); ax.text(x,y,LABEL[st],ha="center",va="center",fontsize=10,fontweight="bold",color=COLORS[st],zorder=7)
for start,end,bend in [("Epi_CA9","Epi_VIM",.16),("Epi_VIM","Epi_JUN",-.14)]: ax.add_patch(FancyArrowPatch(cent.loc[start].to_numpy(),cent.loc[end].to_numpy(),connectionstyle=f"arc3,rad={bend}",arrowstyle="-|>",mutation_scale=14,shrinkA=12,shrinkB=15,lw=2,color="#44515A",alpha=.82,zorder=5))
ax.set_aspect("equal"); ax.axis("off"); cax=fig.add_axes([.25,.065,.5,.025]); cb=fig.colorbar(pts,cax=cax,orientation="horizontal"); cb.set_ticks([low,high],labels=["CA9-like","JUN-like"]); cax.set_title("Cross-sectional epithelial coordinate",fontsize=9,pad=5,color="#263238")
OUT.mkdir(parents=True,exist_ok=True); target=OUT/"Figure2B_epithelial_bridge_umap.pdf"; fig.savefig(target,bbox_inches="tight"); plt.close(fig); print(target)
